In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they exist in current directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to convert timestamp column (seconds) to localized datetimes (UTC->Asia/Shanghai)
def localize_ts(df, ts_col='timestamp'):
    # ensure numeric
    df[ts_col] = pd.to_numeric(df[ts_col], errors='coerce').astype('Int64')
    dt = pd.to_datetime(df[ts_col].astype('int64'), unit='s', utc=True)
    df['_ts_local'] = dt.dt.tz_convert(tz)
    return df

metric_df = localize_ts(metric_df)
trace_df = localize_ts(trace_df)
log_df = localize_ts(log_df)
error_df = localize_ts(error_df)

# Generic summary builder for cmdb/top-name and samples
def build_summary(df, name_col_for_top=None, top_name_label=None, top_message=False):
    total_rows = len(df)
    min_ts = df['_ts_local'].min()
    max_ts = df['_ts_local'].max()
    overview = pd.DataFrame({
        'total_rows': [total_rows],
        'min_timestamp_local': [min_ts],
        'max_timestamp_local': [max_ts]
    })
    # top cmdb_id with counts and earliest/latest timestamps
    cmdb_grp = df.groupby('cmdb_id', dropna=False)['_ts_local'].agg(['count','min','max']).rename(columns={'min':'earliest','max':'latest'}).sort_values('count', ascending=False)
    cmdb_top = cmdb_grp.head(20).reset_index()
    # top name field if provided
    name_top = None
    if name_col_for_top:
        if top_message:
            # for error messages: treat 'message' text as grouping
            name_grp = df.groupby(name_col_for_top, dropna=False).size().rename('count').reset_index().sort_values('count', ascending=False)
            name_top = name_grp.head(20)
        else:
            name_grp = df.groupby(name_col_for_top, dropna=False)['_ts_local'].agg(['count','min']).rename(columns={'min':'earliest'}).sort_values('count', ascending=False)
            name_top = name_grp.head(20).reset_index()
            name_top.columns = [top_name_label if top_name_label else name_col_for_top, 'count', 'earliest']
    samples = df.drop(columns=['_ts_local']).head(5)
    return overview, cmdb_top, name_top, samples

# Build summaries
metric_overview, metric_cmdb_top, metric_kpi_top, metric_samples = build_summary(metric_df, name_col_for_top='kpi_name', top_name_label='kpi_name')
trace_overview, trace_cmdb_top, trace_trace_top, trace_samples = build_summary(trace_df, name_col_for_top='trace_name', top_name_label='trace_name')
log_overview, log_cmdb_top, log_logname_top, log_samples = build_summary(log_df, name_col_for_top='log_name', top_name_label='log_name')
error_overview, error_cmdb_top, error_message_top, error_samples = build_summary(error_df, name_col_for_top='message', top_message=True)

# Keep outputs compact: show only the small summary DataFrames/lists
metric_overview, metric_cmdb_top, metric_kpi_top, metric_samples, \
trace_overview, trace_cmdb_top, trace_trace_top, trace_samples, \
log_overview, log_cmdb_top, log_logname_top, log_samples, \
error_overview, error_cmdb_top, error_message_top, error_samples

```
Out[1]:
```
# Compose a compact plain-English summary using previously loaded variables
# (variables metric_overview, metric_cmdb_top, metric_kpi_top, metric_samples,
#  trace_overview, trace_cmdb_top, trace_trace_top, trace_samples,
#  log_overview, log_cmdb_top, log_logname_top, log_samples,
#  error_overview, error_cmdb_top, error_message_top, error_samples)
# are available in the kernel state from the prior run.

metric_tot = int(metric_overview.loc[0, 'total_rows'])
metric_min = metric_overview.loc[0, 'min_timestamp_local']
metric_max = metric_overview.loc[0, 'max_timestamp_local']
metric_cmdb_top5 = metric_cmdb_top[['cmdb_id','count']].head(5).to_string(index=False)
metric_kpi_top5 = metric_kpi_top[['kpi_name','count']].head(5).to_string(index=False)
metric_sample_rows = metric_samples.head(5).to_dict(orient='records')

trace_tot = int(trace_overview.loc[0, 'total_rows'])
trace_min = trace_overview.loc[0, 'min_timestamp_local']
trace_max = trace_overview.loc[0, 'max_timestamp_local']
trace_cmdb_top5 = trace_cmdb_top[['cmdb_id','count']].head(5).to_string(index=False)
trace_name_top5 = trace_trace_top[['trace_name','count']].head(5).to_string(index=False)
trace_sample_rows = trace_samples.head(5).to_dict(orient='records')

log_tot = int(log_overview.loc[0, 'total_rows'])
log_min = log_overview.loc[0, 'min_timestamp_local']
log_max = log_overview.loc[0, 'max_timestamp_local']
log_cmdb_top5 = log_cmdb_top[['cmdb_id','count']].head(5).to_string(index=False)
log_name_top = log_logname_top[['log_name','count']].head(5).to_string(index=False)
log_sample_rows = log_samples.head(5).to_dict(orient='records')

err_tot = int(error_overview.loc[0, 'total_rows'])
err_min = error_overview.loc[0, 'min_timestamp_local']
err_max = error_overview.loc[0, 'max_timestamp_local']
err_cmdb_top5 = error_cmdb_top[['cmdb_id','count']].head(5).to_string(index=False)
err_msg_top5 = error_message_top.head(5).to_string(index=False)
err_sample_rows = error_samples.head(5).to_dict(orient='records')

summary = (
    f"Compact summary (timestamps in Asia/Shanghai UTC+8):\n\n"
    f"metric.csv:\n"
    f"- total rows: {metric_tot}\n"
    f"- time range: {metric_min} to {metric_max}\n"
    f"- top cmdb_id (top 5):\n{metric_cmdb_top5}\n"
    f"- top kpi_name (top 5):\n{metric_kpi_top5}\n"
    f"- up to 5 sample rows (timestamp, cmdb_id, kpi_name, value): {metric_sample_rows}\n\n"
    f"trace.csv:\n"
    f"- total rows: {trace_tot}\n"
    f"- time range: {trace_min} to {trace_max}\n"
    f"- top cmdb_id (top 5):\n{trace_cmdb_top5}\n"
    f"- top trace_name (top 5):\n{trace_name_top5}\n"
    f"- up to 5 sample rows (timestamp, cmdb_id, trace_name, value): {trace_sample_rows}\n\n"
    f"log.csv:\n"
    f"- total rows: {log_tot}\n"
    f"- time range: {log_min} to {log_max}\n"
    f"- top cmdb_id (top 5):\n{log_cmdb_top5}\n"
    f"- top log_name (top):\n{log_name_top}\n"
    f"- up to 5 sample rows (timestamp, cmdb_id, log_name, value): {log_sample_rows}\n\n"
    f"error_logs.csv:\n"
    f"- total rows: {err_tot}\n"
    f"- time range: {err_min} to {err_max}\n"
    f"- top cmdb_id (top 5):\n{err_cmdb_top5}\n"
    f"- top distinct messages (top 5 by count):\n{err_msg_top5}\n"
    f"- up to 5 sample rows (timestamp, cmdb_id, message): {err_sample_rows}\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows       min_timestamp_local       max_timestamp_local
0      146292 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00,                      cmdb_id  count                  earliest                    latest
0                  adservice  10230 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
1                 adservice2   9990 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
2                frontend2-0   5040 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
3                 frontend-0   4920 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
4                 frontend-1   4680 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
5                 frontend-2   4500 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
6          checkoutservice-0   3900 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
7          checkoutservice-2   3660 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
8          checkoutservice-1   3540 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
9    productcatalogservice-0   3480 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
10   productcatalogservice-2   3480 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
11   productcatalogservice-1   3360 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
12        checkoutservice2-0   3300 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
13              adservice2-0   3060 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
14             cartservice-1   3000 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
15  productcatalogservice2-0   2880 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
16             cartservice-0   2760 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
17             cartservice-2   2640 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
18            cartservice2-0   2640 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
19               adservice-0   2580 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00,                                              kpi_name  count                  earliest
0   container.node-5.container_network_transmit_pa...   1050 2022-03-21 03:00:00+08:00
1   container.node-5.container_network_transmit_pa...   1050 2022-03-21 03:00:00+08:00
2   container.node-5.container_network_transmit_MB...   1050 2022-03-21 03:00:00+08:00
3   container.node-5.container_network_transmit_er...   1050 2022-03-21 03:00:00+08:00
4   container.node-5.container_network_receive_pac...   1050 2022-03-21 03:00:00+08:00
5   container.node-5.container_network_receive_pac...   1050 2022-03-21 03:00:00+08:00
6   container.node-5.container_network_receive_MB....   1050 2022-03-21 03:00:00+08:00
7   container.node-5.container_network_receive_err...   1050 2022-03-21 03:00:00+08:00
8           container.node-5.container_spec_cpu_quota    990 2022-03-21 03:00:00+08:00
9          container.node-5.container_spec_cpu_shares    990 2022-03-21 03:00:00+08:00
10              container.node-5.container_memory_rss    990 2022-03-21 03:00:00+08:00
11             container.node-5.container_memory_swap    990 2022-03-21 03:00:00+08:00
12  container.node-5.container_spec_memory_reserva...    990 2022-03-21 03:00:00+08:00
13    container.node-5.container_spec_memory_limit_MB    990 2022-03-21 03:00:00+08:00
14     container.node-5.container_tasks_state.stopped    990 2022-03-21 03:00:00+08:00
15    container.node-5.container_tasks_state.sleeping    990 2022-03-21 03:00:00+08:00
16         container.node-5.container_memory_usage_MB    990 2022-03-21 03:00:00+08:00
17                 container.node-5.container_sockets    990 2022-03-21 03:00:00+08:00
18  container.node-5.container_spec_memory_swap_li...    990 2022-03-21 03:00:00+08:00
19      container.node-5.container_start_time_seconds    990 2022-03-21 03:00:00+08:00,     timestamp    cmdb_id        kpi_name       value
0  1647802800  adservice  app.grpc.count   82.000000
1  1647802800  adservice    app.grpc.mrt    0.002577
2  1647802800  adservice     app.grpc.rr  100.000000
3  1647802800  adservice     app.grpc.sr   75.609756
4  1647802800  adservice  app.http.count   24.000000,    total_rows       min_timestamp_local       max_timestamp_local
0       29328 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00,                     cmdb_id  count                  earliest                    latest
0                frontend-2   2568 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
1                frontend-0   2492 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
2                frontend-1   2484 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
3         checkoutservice-1   1580 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
4         checkoutservice-0   1532 2022-03-21 03:01:00+08:00 2022-03-21 03:29:00+08:00
5         checkoutservice-2   1484 2022-03-21 03:01:00+08:00 2022-03-21 03:29:00+08:00
6   productcatalogservice-1   1076 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
7               frontend2-0   1060 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
8   productcatalogservice-0   1044 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
9   productcatalogservice-2   1044 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
10       checkoutservice2-0    936 2022-03-21 03:00:00+08:00 2022-03-21 03:28:00+08:00
11  recommendationservice-0    840 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
12  recommendationservice-2    840 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
13  recommendationservice-1    840 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
14            cartservice-1    716 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
15            cartservice-0    708 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
16            cartservice-2    684 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
17        currencyservice-1    624 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
18        currencyservice-0    620 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
19        currencyservice-2    616 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00,                                     trace_name  count                  earliest
0          trace.from_frontend-2.duration_mean    582 2022-03-21 03:00:00+08:00
1           trace.from_frontend-2.duration_p95    582 2022-03-21 03:00:00+08:00
2             trace.from_frontend-2.error_rate    582 2022-03-21 03:00:00+08:00
3              trace.from_frontend-2.row_count    582 2022-03-21 03:00:00+08:00
4          trace.from_frontend-0.duration_mean    563 2022-03-21 03:00:00+08:00
5           trace.from_frontend-0.duration_p95    563 2022-03-21 03:00:00+08:00
6             trace.from_frontend-0.error_rate    563 2022-03-21 03:00:00+08:00
7              trace.from_frontend-0.row_count    563 2022-03-21 03:00:00+08:00
8              trace.from_frontend-1.row_count    561 2022-03-21 03:00:00+08:00
9             trace.from_frontend-1.error_rate    561 2022-03-21 03:00:00+08:00
10          trace.from_frontend-1.duration_p95    561 2022-03-21 03:00:00+08:00
11         trace.from_frontend-1.duration_mean    561 2022-03-21 03:00:00+08:00
12                        trace.self.row_count    453 2022-03-21 03:00:00+08:00
13                       trace.self.error_rate    453 2022-03-21 03:00:00+08:00
14                     trace.self.duration_p95    453 2022-03-21 03:00:00+08:00
15                    trace.self.duration_mean    453 2022-03-21 03:00:00+08:00
16  trace.from_checkoutservice-1.duration_mean    312 2022-03-21 03:00:00+08:00
17   trace.from_checkoutservice-1.duration_p95    312 2022-03-21 03:00:00+08:00
18     trace.from_checkoutservice-1.error_rate    312 2022-03-21 03:00:00+08:00
19      trace.from_checkoutservice-1.row_count    312 2022-03-21 03:00:00+08:00,     timestamp      cmdb_id                           trace_name     value
0  1647802800  adservice-0  trace.from_frontend-0.duration_mean  0.000022
1  1647802800  adservice-0   trace.from_frontend-0.duration_p95  0.000040
2  1647802800  adservice-0     trace.from_frontend-0.error_rate  0.000000
3  1647802800  adservice-0      trace.from_frontend-0.row_count  4.000000
4  1647802800  adservice-0  trace.from_frontend-1.duration_mean  0.000013,    total_rows       min_timestamp_local       max_timestamp_local
0        1728 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00,                     cmdb_id  count                  earliest                    latest
0               adservice-0     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
1               adservice-1     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
2               adservice-2     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
3             cartservice-0     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
4             cartservice-1     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
5             cartservice-2     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
6         currencyservice-1     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
7         currencyservice-2     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
8         currencyservice-0     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
9   recommendationservice-1     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
10  productcatalogservice-0     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
11  productcatalogservice-1     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
12               frontend-2     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
13               frontend-1     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
14               frontend-0     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
15        shippingservice-0     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
16        shippingservice-1     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
17        shippingservice-2     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
18  recommendationservice-2     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00
19  recommendationservice-0     60 2022-03-21 03:00:00+08:00 2022-03-21 03:29:00+08:00,           log_name  count                  earliest
0  log.error_count    864 2022-03-21 03:00:00+08:00
1    log.row_count    864 2022-03-21 03:00:00+08:00,     timestamp      cmdb_id         log_name  value
0  1647802800  adservice-0  log.error_count    0.0
1  1647802800  adservice-0    log.row_count  476.0
2  1647802800  adservice-1  log.error_count    0.0
3  1647802800  adservice-1    log.row_count   46.0
4  1647802800  adservice-2  log.error_count    0.0,    total_rows       min_timestamp_local       max_timestamp_local
0         544 2022-03-21 03:00:05+08:00 2022-03-21 03:29:56+08:00,       cmdb_id  count                  earliest                    latest
0  frontend-2    232 2022-03-21 03:00:05+08:00 2022-03-21 03:29:53+08:00
1  frontend-0    158 2022-03-21 03:00:07+08:00 2022-03-21 03:29:31+08:00
2  frontend-1    154 2022-03-21 03:00:06+08:00 2022-03-21 03:29:56+08:00,                                              message  count
0  severity: warning, message: failed to retrieve...    544,     timestamp     cmdb_id                                            message
0  1647802805  frontend-2  severity: warning, message: failed to retrieve...
1  1647802806  frontend-1  severity: warning, message: failed to retrieve...
2  1647802807  frontend-0  severity: warning, message: failed to retrieve...
3  1647802810  frontend-2  severity: warning, message: failed to retrieve...
4  1647802814  frontend-1  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df if available in kernel, otherwise load
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')

# ensure timezone and timestamp column exist
tz = pytz.timezone('Asia/Shanghai')
if '_ts_local' not in metric_df.columns:
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    dt = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True)
    metric_df['_ts_local'] = dt.dt.tz_convert(tz)

# ensure value is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series (BEFORE filtering)
global_thresh = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(global_p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan,
         global_p5 = lambda x: float(x.dropna().quantile(0.05)) if x.dropna().size>0 else np.nan)
    .reset_index()
)

# 2) Filter metric rows to incident window (inclusive)
start = pd.Timestamp('2022-03-21 03:00:00', tz=tz)
end   = pd.Timestamp('2022-03-21 03:30:00', tz=tz)
window_df = metric_df[(metric_df['_ts_local'] >= start) & (metric_df['_ts_local'] <= end)].copy()

# 3) Merge global thresholds into window rows so comparisons use global p95/p5
window_df = window_df.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')

# flags for anomalies relative to global thresholds (strict > / < as requested)
window_df['above_p95'] = (window_df['value'] > window_df['global_p95']).astype('int')
window_df['below_p5']  = (window_df['value'] < window_df['global_p5']).astype('int')

# aggregate per (cmdb_id, kpi_name)
agg_basic = (
    window_df
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        total_points_in_window = ('value','count'),
        anomalies_above_p95_count = ('above_p95','sum'),
        anomalies_below_p5_count = ('below_p5','sum'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
    )
    .reset_index()
)

# earliest timestamps for above/below (or NaT)
earliest_above = (
    window_df[window_df['above_p95']==1]
    .groupby(['cmdb_id','kpi_name'], dropna=False)['_ts_local']
    .min()
    .rename('earliest_above_p95_timestamp_local')
    .reset_index()
)
earliest_below = (
    window_df[window_df['below_p5']==1]
    .groupby(['cmdb_id','kpi_name'], dropna=False)['_ts_local']
    .min()
    .rename('earliest_below_p5_timestamp_local')
    .reset_index()
)

# merge earliest timestamps
agg = agg_basic.merge(earliest_above, on=['cmdb_id','kpi_name'], how='left')
agg = agg.merge(earliest_below, on=['cmdb_id','kpi_name'], how='left')

# merge global thresholds (so each aggregated row has global_p95/global_p5)
agg = agg.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')

# convert counts to int
agg['anomalies_above_p95_count'] = agg['anomalies_above_p95_count'].astype(int)
agg['anomalies_below_p5_count'] = agg['anomalies_below_p5_count'].astype(int)
agg['total_points_in_window'] = agg['total_points_in_window'].astype(int)

# 4) Filter rows where either anomaly count > 0
agg['total_anomalies'] = agg['anomalies_above_p95_count'] + agg['anomalies_below_p5_count']
result_df = agg[(agg['anomalies_above_p95_count']>0) | (agg['anomalies_below_p5_count']>0)].copy()

# sort by total anomaly count desc, then by total_points_in_window desc as tiebreaker
result_df = result_df.sort_values(['total_anomalies','total_points_in_window'], ascending=[False, False])

# select and order required columns
final_cols = [
    'cmdb_id','kpi_name','global_p95','global_p5',
    'total_points_in_window','anomalies_above_p95_count','earliest_above_p95_timestamp_local',
    'max_value_in_window','anomalies_below_p5_count','earliest_below_p5_timestamp_local','min_value_in_window'
]
final_table = result_df[final_cols].head(20).reset_index(drop=True)

# Keep final output compact
final_table

```
Out[2]:
```
Summary of results (computed as requested):

- Window and method: global P95/P5 were computed across the entire metric series (not window-filtered). The incident window filtered was 2022-03-21 03:00:00+08:00 through 2022-03-21 03:30:00+08:00.

- Output scope: top 20 (cmdb_id, kpi_name) rows with anomalies in the window (rows where anomalies_above_p95_count > 0 or anomalies_below_p5_count > 0), sorted by total anomaly count. All 20 returned rows are for cmdb_id = adservice.

- Common pattern: each returned (adservice, kpi_name) has 30 measurements in the window and shows 2 points above the global P95 and 2 points below the global P5.

- Example entries (showing global P95, global P5, window counts and extremes):
  - app.grpc.count — global_p95 88.55, global_p5 69.45; total_points_in_window 30; anomalies_above_p95_count 2 (earliest 2022-03-21 03:08+08); max_in_window 91.0; anomalies_below_p5_count 2 (earliest 2022-03-21 03:07+08); min_in_window 66.0.
  - app.grpc.mrt — global_p95 0.0026123, global_p5 0.0023384; total 30; 2 above (earliest 03:24+08), max 0.0045955; 2 below (earliest 03:09+08), min 0.0022507.
  - runtime.java_lang_Memory_HeapMemoryUsage_used — global_p95 21,895,580; global_p5 20,733,130; total 30; 2 above (earliest 03:04+08), max ~22,007,230; 2 below (earliest 03:10+08), min ~20,647,290.

- Other notable KPIs in the top-20: many JVM runtime / GarbageCollector / MemoryPool metrics and process CPU load metrics all show the same anomaly counts (2 above P95 and 2 below P5) with differing global thresholds and earliest anomaly timestamps.

- Conclusion: The top-20 anomaly hits during the incident window are concentrated on adservice metrics (multiple JVM memory and GC KPIs plus gRPC/http KPIs). Each affected metric shows a small number of high and low excursions relative to its global distribution (2 above global P95 and 2 below global P5 in the window).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_p95     global_p5  total_points_in_window  anomalies_above_p95_count earliest_above_p95_timestamp_local  max_value_in_window  anomalies_below_p5_count earliest_below_p5_timestamp_local  min_value_in_window
0   adservice                                     app.grpc.count  8.855000e+01  6.945000e+01                      30                          2          2022-03-21 03:08:00+08:00         9.100000e+01                         2         2022-03-21 03:07:00+08:00         6.600000e+01
1   adservice                                       app.grpc.mrt  2.612301e-03  2.338365e-03                      30                          2          2022-03-21 03:24:00+08:00         4.595455e-03                         2         2022-03-21 03:09:00+08:00         2.250725e-03
2   adservice                                        app.grpc.sr  8.003914e+01  7.365240e+01                      30                          2          2022-03-21 03:22:00+08:00         8.055556e+01                         2         2022-03-21 03:17:00+08:00         7.142857e+01
3   adservice                                       app.http.mrt  9.739375e-03  9.181250e-03                      30                          2          2022-03-21 03:02:00+08:00         9.885417e-03                         2         2022-03-21 03:08:00+08:00         9.079167e-03
4   adservice  runtime.java_lang_GarbageCollector_CollectionC...  4.161205e+04  4.138936e+04                      30                          2          2022-03-21 03:28:00+08:00         4.162433e+04                         2         2022-03-21 03:00:00+08:00         4.137692e+04
5   adservice  runtime.java_lang_GarbageCollector_CollectionT...  6.536265e+04  6.501477e+04                      30                          2          2022-03-21 03:28:00+08:00         6.538200e+04                         2         2022-03-21 03:00:00+08:00         6.499567e+04
6   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.813981e+08  2.798324e+08                      30                          2          2022-03-21 03:28:00+08:00         2.814847e+08                         2         2022-03-21 03:00:00+08:00         2.797451e+08
7   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  4.161205e+04  4.138936e+04                      30                          2          2022-03-21 03:28:00+08:00         4.162433e+04                         2         2022-03-21 03:00:00+08:00         4.137692e+04
8   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  1.829993e+05  1.235855e+05                      30                          2          2022-03-21 03:02:00+08:00         1.898060e+05                         2         2022-03-21 03:07:00+08:00         1.128587e+05
9   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  1.657248e+07  1.657048e+07                      30                          2          2022-03-21 03:28:00+08:00         1.657267e+07                         2         2022-03-21 03:00:00+08:00         1.657029e+07
10  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.364502e+05  1.468399e+05                      30                          2          2022-03-21 03:22:00+08:00         2.529433e+05                         2         2022-03-21 03:10:00+08:00         1.440647e+05
11  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  1.657247e+07  1.657047e+07                      30                          2          2022-03-21 03:28:00+08:00         1.657267e+07                         2         2022-03-21 03:00:00+08:00         1.657026e+07
12  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...  2.813981e+08  2.798324e+08                      30                          2          2022-03-21 03:28:00+08:00         2.814847e+08                         2         2022-03-21 03:00:00+08:00         2.797451e+08
13  adservice  runtime.java_lang_MemoryPool_CollectionUsage_u...  2.453922e+05  1.336361e+05                      30                          2          2022-03-21 03:22:00+08:00         2.617967e+05                         2         2022-03-21 03:10:00+08:00         1.220260e+05
14  adservice  runtime.java_lang_MemoryPool_PeakUsage_used.Te...  1.657248e+07  1.657048e+07                      30                          2          2022-03-21 03:28:00+08:00         1.657267e+07                         2         2022-03-21 03:00:00+08:00         1.657029e+07
15  adservice  runtime.java_lang_MemoryPool_Usage_used.Eden_S...  5.067650e+06  3.715603e+06                      30                          2          2022-03-21 03:14:00+08:00         5.134908e+06                         2         2022-03-21 03:09:00+08:00         3.453847e+06
16  adservice  runtime.java_lang_MemoryPool_Usage_used.Surviv...  2.453922e+05  1.336361e+05                      30                          2          2022-03-21 03:22:00+08:00         2.617967e+05                         2         2022-03-21 03:10:00+08:00         1.220260e+05
17  adservice  runtime.java_lang_MemoryPool_Usage_used.Tenure...  1.657248e+07  1.657048e+07                      30                          2          2022-03-21 03:28:00+08:00         1.657267e+07                         2         2022-03-21 03:00:00+08:00         1.657029e+07
18  adservice      runtime.java_lang_Memory_HeapMemoryUsage_used  2.189558e+07  2.073313e+07                      30                          2          2022-03-21 03:04:00+08:00         2.200723e+07                         2         2022-03-21 03:10:00+08:00         2.064729e+07
19  adservice   runtime.java_lang_OperatingSystem_ProcessCpuLoad  1.732685e-04  1.568073e-04                      30                          2          2022-03-21 03:09:00+08:00         1.777670e-04                         2         2022-03-21 03:13:00+08:00         1.463260e-04```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df if present
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')

# timezone
tz = pytz.timezone('Asia/Shanghai')

# ensure timestamp columns and types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
if '_ts_local' not in metric_df.columns:
    dt = pd.to_datetime(metric_df['timestamp'].astype('int64'), unit='s', utc=True)
    metric_df['_ts_local'] = dt.dt.tz_convert(tz)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using the full series (must be before filtering)
global_thresh = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(global_p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan,
         global_p5 = lambda x: float(x.dropna().quantile(0.05)) if x.dropna().size>0 else np.nan)
    .reset_index()
)

# 2) Filter to incident window (inclusive)
start = pd.Timestamp('2022-03-21 03:00:00', tz=tz)
end   = pd.Timestamp('2022-03-21 03:30:00', tz=tz)
window_df = metric_df[(metric_df['_ts_local'] >= start) & (metric_df['_ts_local'] <= end)].copy()

# 3) Merge thresholds and mark anomalies
window_df = window_df.merge(global_thresh, on=['cmdb_id','kpi_name'], how='left')
window_df['anomalous_above'] = window_df['value'] > window_df['global_p95']
window_df['anomalous_below'] = window_df['value'] < window_df['global_p5']
window_df['is_anomalous'] = window_df['anomalous_above'] | window_df['anomalous_below']

# compute severity for ordering top anomalies (absolute deviation from nearest threshold)
def compute_severity(row):
    if row['anomalous_above']:
        return float(row['value'] - row['global_p95'])
    if row['anomalous_below']:
        return float(row['global_p5'] - row['value'])
    return 0.0

window_df['anomaly_severity'] = window_df.apply(compute_severity, axis=1)

# anomalies subset
anomalies_df = window_df[window_df['is_anomalous']].copy()

# 4) Aggregate anomalous rows by (cmdb_id, timestamp_local)
# total_kpi_points_at_that_timestamp: count of all metric kpis recorded for that cmdb_id at that timestamp in the window
total_points = (
    window_df
    .groupby(['cmdb_id','_ts_local'], dropna=False)
    .size()
    .rename('total_kpi_points_at_timestamp')
    .reset_index()
)

# build grouped anomalies info: anomaly_count (distinct kpi_name), top_5_kpis_anomalous and sample_values
def summarize_group(g):
    cmdb = g.name[0]
    ts = g.name[1]
    # distinct kpis anomalous count
    anomaly_count = g['kpi_name'].nunique()
    # pick top kpis by severity (if multiple rows per kpi at same ts, take max severity and corresponding value)
    by_kpi = (g.groupby('kpi_name', dropna=False)
               .apply(lambda df: pd.Series({
                   'severity': df['anomaly_severity'].max(),
                   'value': df.loc[df['anomaly_severity'].idxmax(),'value']
               }))
               .reset_index())
    by_kpi_sorted = by_kpi.sort_values('severity', ascending=False)
    top_kpis = by_kpi_sorted.head(5)
    top_5_kpis_anomalous = ",".join(top_kpis['kpi_name'].astype(str).tolist())
    # sample_values as "kpi:value" comma-separated for the top kpis
    sample_values = ",".join([f"{k}:{v}" for k,v in zip(top_kpis['kpi_name'], top_kpis['value'])])
    return pd.Series({
        'anomaly_count': int(anomaly_count),
        'top_5_kpis_anomalous': top_5_kpis_anomalous,
        'sample_values': sample_values
    })

# group anomalies by cmdb_id and timestamp
if not anomalies_df.empty:
    grouped = anomalies_df.groupby(['cmdb_id','_ts_local'], dropna=False)
    anomalies_grouped = grouped.apply(summarize_group).reset_index()
    # merge total points
    anomalies_grouped = anomalies_grouped.merge(total_points, on=['cmdb_id','_ts_local'], how='left')
    # rename timestamp column to timestamp_local for output
    anomalies_grouped = anomalies_grouped.rename(columns={'_ts_local':'timestamp_local',
                                                          'total_kpi_points_at_timestamp':'total_kpi_points_at_that_timestamp'})
    # sort by anomaly_count desc, then total_kpi_points_at_that_timestamp desc
    anomalies_by_ts = anomalies_grouped.sort_values(['anomaly_count','total_kpi_points_at_that_timestamp'], ascending=[False, False]).reset_index(drop=True)
else:
    # empty DataFrame with desired columns
    anomalies_by_ts = pd.DataFrame(columns=['cmdb_id','timestamp_local','anomaly_count','top_5_kpis_anomalous','sample_values','total_kpi_points_at_that_timestamp'])

# 5) Per-cmdb_id summary: total_anomaly_count_in_window and earliest_anomaly_timestamp_local
if not anomalies_df.empty:
    per_cmdb = (
        anomalies_df
        .groupby('cmdb_id', dropna=False)
        .agg(
            total_anomaly_count_in_window = ('kpi_name','count'),
            earliest_anomaly_timestamp_local = ('_ts_local','min')
        )
        .reset_index()
        .sort_values('total_anomaly_count_in_window', ascending=False)
    )
else:
    per_cmdb = pd.DataFrame(columns=['cmdb_id','total_anomaly_count_in_window','earliest_anomaly_timestamp_local'])

# Keep outputs compact: top-20 anomalies_by_ts and top-10 per_cmdb
anomalies_by_ts_head = anomalies_by_ts.head(20).copy()
per_cmdb_head = per_cmdb.head(10).copy()

# expose intermediate tables for later steps
_global_thresh_for_next_steps = global_thresh
_window_df_for_next_steps = window_df
_anomalies_df_for_next_steps = anomalies_df
_anomalies_by_ts_for_next_steps = anomalies_by_ts
_per_cmdb_summary_for_next_steps = per_cmdb

# display compact results (use variable names as required)
anomalies_by_ts_head, per_cmdb_head

```
Out[3]:
```
```python
# Compose a compact plain-English summary from prior results available in the kernel:
# anomalies_by_ts_head (top-20 aggregated anomaly-by-timestamp) and per_cmdb_head (top-10 per-cmdb summary)

rows = []
for i, r in anomalies_by_ts_head.head(5).iterrows():
    rows.append(f"{i+1}. {r['cmdb_id']} @ {r['timestamp_local'].tz_convert('Asia/Shanghai')} — anomaly_count={int(r['anomaly_count'])}, total_kpi_points={int(r['total_kpi_points_at_that_timestamp'])}")

cmdb_rows = []
for i, r in per_cmdb_head.head(5).iterrows():
    ts = r['earliest_anomaly_timestamp_local']
    cmdb_rows.append(f"{i+1}. {r['cmdb_id']} — total_anomaly_count={int(r['total_anomaly_count_in_window'])}, earliest={ts.tz_convert('Asia/Shanghai')}")

summary = (
    "Summary of anomalies (values outside global P95/P5) in 2022-03-21 03:00–03:30 (Asia/Shanghai):\n\n"
    "Top timestamp-level anomaly hotspots (top 5):\n" +
    "\n".join(rows) + "\n\n"
    "Top cmdb_id by total anomalies in window (top 5):\n" +
    "\n".join(cmdb_rows) + "\n\n"
    "Notes:\n"
    "- The single largest timestamp-level hotspot is checkoutservice-2 at 2022-03-21 03:26:00+08:00 with 50 distinct KPI anomalies (122 KPI points recorded for that cmdb_id at that minute).\n"
    "- Other high-count hotspots include checkoutservice-0 (03:02, 49 anomalies), checkoutservice-2 (03:16, 49), and several frontend instances with 30+ anomalies at specific minutes.\n"
    "- Per-cmdb totals show frontends as the top affected components (e.g., frontend-0: 258 anomalies, frontend-2: 253, frontend2-0: 245, frontend-1: 228), followed by checkoutservice instances and adservice2.\n"
    "- Earliest anomalies for the top cmdb_ids occur at the start of the window (2022-03-21 03:00:00+08:00) for many components.\n\n"
    "These results are compact summaries derived from metric.csv using global P95/P5 computed from the entire metric series."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id           timestamp_local  anomaly_count                               top_5_kpis_anomalous                                      sample_values  total_kpi_points_at_that_timestamp
0    checkoutservice-2 2022-03-21 03:26:00+08:00             50  mesh.source.checkoutservice.jaeger-collector.i...  mesh.source.checkoutservice.jaeger-collector.i...                                 122
1    checkoutservice-0 2022-03-21 03:02:00+08:00             49  mesh.source.checkoutservice.jaeger-collector.i...  mesh.source.checkoutservice.jaeger-collector.i...                                 130
2    checkoutservice-2 2022-03-21 03:16:00+08:00             49  mesh.source.checkoutservice.jaeger-collector.i...  mesh.source.checkoutservice.jaeger-collector.i...                                 122
3    checkoutservice-0 2022-03-21 03:20:00+08:00             47  mesh.source.checkoutservice.jaeger-collector.i...  mesh.source.checkoutservice.jaeger-collector.i...                                 130
4    checkoutservice-2 2022-03-21 03:06:00+08:00             47  mesh.source.checkoutservice.jaeger-collector.i...  mesh.source.checkoutservice.jaeger-collector.i...                                 122
5    checkoutservice-0 2022-03-21 03:12:00+08:00             46  mesh.source.checkoutservice.jaeger-collector.i...  mesh.source.checkoutservice.jaeger-collector.i...                                 130
6           frontend-1 2022-03-21 03:16:00+08:00             38  mesh.destination.unknown.frontend.istio_respon...  mesh.destination.unknown.frontend.istio_respon...                                 156
7           adservice2 2022-03-21 03:00:00+08:00             36  runtime.java_lang_OperatingSystem_ProcessCpuTi...  runtime.java_lang_OperatingSystem_ProcessCpuTi...                                 333
8           frontend-0 2022-03-21 03:20:00+08:00             34  mesh.source.frontend.jaeger-collector.istio_re...  mesh.source.frontend.jaeger-collector.istio_re...                                 164
9          frontend2-0 2022-03-21 03:27:00+08:00             33  mesh.source.frontend2.jaeger-collector.istio_r...  mesh.source.frontend2.jaeger-collector.istio_r...                                 168
10          frontend-0 2022-03-21 03:17:00+08:00             33  mesh.source.frontend.productcatalogservice.ist...  mesh.source.frontend.productcatalogservice.ist...                                 164
11  checkoutservice2-0 2022-03-21 03:06:00+08:00             33  mesh.source.checkoutservice2.currencyservice2....  mesh.source.checkoutservice2.currencyservice2....                                 110
12          adservice2 2022-03-21 03:29:00+08:00             29  runtime.java_lang_OperatingSystem_ProcessCpuTi...  runtime.java_lang_OperatingSystem_ProcessCpuTi...                                 333
13          frontend-1 2022-03-21 03:06:00+08:00             29  mesh.source.frontend.jaeger-collector.istio_re...  mesh.source.frontend.jaeger-collector.istio_re...                                 156
14          adservice2 2022-03-21 03:01:00+08:00             27  runtime.java_lang_OperatingSystem_ProcessCpuTi...  runtime.java_lang_OperatingSystem_ProcessCpuTi...                                 333
15          frontend-0 2022-03-21 03:12:00+08:00             26  mesh.source.frontend.adservice.istio_request_b...  mesh.source.frontend.adservice.istio_request_b...                                 164
16          frontend-2 2022-03-21 03:01:00+08:00             25  container.node-5.container_memory_rss,mesh.sou...  container.node-5.container_memory_rss:20297728...                                 150
17           adservice 2022-03-21 03:28:00+08:00             24  runtime.java_lang_OperatingSystem_ProcessCpuTi...  runtime.java_lang_OperatingSystem_ProcessCpuTi...                                 341
18          frontend-0 2022-03-21 03:26:00+08:00             24  mesh.source.frontend.productcatalogservice.ist...  mesh.source.frontend.productcatalogservice.ist...                                 164
19          frontend-1 2022-03-21 03:22:00+08:00             24  mesh.source.frontend.adservice.istio_request_b...  mesh.source.frontend.adservice.istio_request_b...                                 156,                     cmdb_id  total_anomaly_count_in_window earliest_anomaly_timestamp_local
27               frontend-0                            258        2022-03-21 03:00:00+08:00
29               frontend-2                            253        2022-03-21 03:00:00+08:00
30              frontend2-0                            245        2022-03-21 03:00:00+08:00
28               frontend-1                            228        2022-03-21 03:00:00+08:00
14        checkoutservice-2                            191        2022-03-21 03:00:00+08:00
12        checkoutservice-0                            186        2022-03-21 03:00:00+08:00
4                adservice2                            177        2022-03-21 03:00:00+08:00
45  productcatalogservice-2                            156        2022-03-21 03:00:00+08:00
15       checkoutservice2-0                            152        2022-03-21 03:00:00+08:00
44  productcatalogservice-1                            152        2022-03-21 03:00:00+08:00)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse trace_df if available
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# ensure timestamp/local column
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
if '_ts_local' not in trace_df.columns:
    dt = pd.to_datetime(trace_df['timestamp'].astype('int64'), unit='s', utc=True)
    trace_df['_ts_local'] = dt.dt.tz_convert(tz)

trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) global P95 and P5 per (cmdb_id, trace_name) using full series (BEFORE filtering)
global_thresh_trace = (
    trace_df
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(
        global_p95 = lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan,
        global_p5  = lambda x: float(x.dropna().quantile(0.05)) if x.dropna().size>0 else np.nan
    )
    .reset_index()
)

# 2) filter to incident window (inclusive)
start = pd.Timestamp('2022-03-21 03:00:00', tz=tz)
end   = pd.Timestamp('2022-03-21 03:30:00', tz=tz)
window_trace_df = trace_df[(trace_df['_ts_local'] >= start) & (trace_df['_ts_local'] <= end)].copy()

# 3) merge thresholds and mark anomalies with special rules
window_trace_df = window_trace_df.merge(global_thresh_trace, on=['cmdb_id','trace_name'], how='left')

# classify trace types
window_trace_df['is_row_count'] = window_trace_df['trace_name'].str.contains('row_count', na=False)
window_trace_df['is_error_rate'] = window_trace_df['trace_name'].str.contains('error_rate', na=False)
window_trace_df['is_duration'] = window_trace_df['trace_name'].str.contains('duration', na=False)

# baseline flags
window_trace_df['anomalous_above'] = False
window_trace_df['anomalous_below'] = False

# apply rules:
# row_count: anomaly if value < global_p5
mask_row = window_trace_df['is_row_count'] & window_trace_df['value'].notna()
window_trace_df.loc[mask_row, 'anomalous_below'] = window_trace_df.loc[mask_row, 'value'] < window_trace_df.loc[mask_row, 'global_p5']

# error_rate: anomaly if value > global_p95
mask_err = window_trace_df['is_error_rate'] & window_trace_df['value'].notna()
window_trace_df.loc[mask_err, 'anomalous_above'] = window_trace_df.loc[mask_err, 'value'] > window_trace_df.loc[mask_err, 'global_p95']

# duration: anomaly if value > global_p95
mask_dur = window_trace_df['is_duration'] & window_trace_df['value'].notna()
window_trace_df.loc[mask_dur, 'anomalous_above'] = window_trace_df.loc[mask_dur, 'value'] > window_trace_df.loc[mask_dur, 'global_p95']

# other traces: mark both sides
mask_other = ~(window_trace_df['is_row_count'] | window_trace_df['is_error_rate'] | window_trace_df['is_duration']) & window_trace_df['value'].notna()
window_trace_df.loc[mask_other, 'anomalous_above'] = window_trace_df.loc[mask_other, 'value'] > window_trace_df.loc[mask_other, 'global_p95']
window_trace_df.loc[mask_other, 'anomalous_below'] = window_trace_df.loc[mask_other, 'value'] < window_trace_df.loc[mask_other, 'global_p5']

window_trace_df['is_anomalous'] = window_trace_df['anomalous_above'] | window_trace_df['anomalous_below']

# compute severity for ordering: distance from threshold (use absolute)
def sev(row):
    if row['anomalous_above']:
        return float(row['value'] - row['global_p95'])
    if row['anomalous_below']:
        return float(row['global_p5'] - row['value'])
    return 0.0

window_trace_df['severity'] = window_trace_df.apply(sev, axis=1)

# subset anomalies
anomalies_trace_df = window_trace_df[window_trace_df['is_anomalous']].copy()

# helper: mark origin-from-frontend and downstream cmdb priority
downstream_services = ['checkoutservice','adservice','productcatalogservice','recommendationservice','paymentservice','cartservice','shippingservice','currencyservice','emailservice']
anomalies_trace_df['from_frontend'] = anomalies_trace_df['trace_name'].str.contains('from_frontend', na=False)
anomalies_trace_df['is_downstream_cmdb'] = anomalies_trace_df['cmdb_id'].astype(str).apply(lambda x: any(x.startswith(s) for s in downstream_services))

# 4A) Aggregate by (cmdb_id, trace_name)
if not anomalies_trace_df.empty:
    agg_pairs = (
        window_trace_df.groupby(['cmdb_id','trace_name'], dropna=False)
        .agg(
            total_points_in_window = ('value','count'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min')
        )
        .reset_index()
    )

    above = (anomalies_trace_df[anomalies_trace_df['anomalous_above']]
             .groupby(['cmdb_id','trace_name'], dropna=False)['_ts_local']
             .agg(['count','min'])
             .rename(columns={'count':'anomalies_above_p95_count','min':'earliest_above_p95_timestamp_local'})
             .reset_index()
            )
    below = (anomalies_trace_df[anomalies_trace_df['anomalous_below']]
             .groupby(['cmdb_id','trace_name'], dropna=False)['_ts_local']
             .agg(['count','min'])
             .rename(columns={'count':'anomalies_below_p5_count','min':'earliest_below_p5_timestamp_local'})
             .reset_index()
            )

    pairs = agg_pairs.merge(global_thresh_trace, on=['cmdb_id','trace_name'], how='left')
    pairs = pairs.merge(above, on=['cmdb_id','trace_name'], how='left')
    pairs = pairs.merge(below, on=['cmdb_id','trace_name'], how='left')

    # fill NaN counts with 0
    pairs['anomalies_above_p95_count'] = pairs['anomalies_above_p95_count'].fillna(0).astype(int)
    pairs['anomalies_below_p5_count'] = pairs['anomalies_below_p5_count'].fillna(0).astype(int)

    pairs['total_anomalies'] = pairs['anomalies_above_p95_count'] + pairs['anomalies_below_p5_count']

    # keep only pairs with any anomalies
    pairs = pairs[pairs['total_anomalies'] > 0].copy()

    # add priority flags to prioritize frontend-origin traces and downstream cmdbs
    pairs['from_frontend'] = pairs['trace_name'].str.contains('from_frontend', na=False)
    pairs['is_downstream_cmdb'] = pairs['cmdb_id'].astype(str).apply(lambda x: any(x.startswith(s) for s in downstream_services))

    # sort: primary by total_anomalies desc, then from_frontend, then is_downstream_cmdb
    pairs = pairs.sort_values(['total_anomalies','from_frontend','is_downstream_cmdb'], ascending=[False, False, False])

    top_pairs = pairs[['cmdb_id','trace_name','global_p95','global_p5','total_points_in_window',
                       'anomalies_above_p95_count','earliest_above_p95_timestamp_local',
                       'anomalies_below_p5_count','earliest_below_p5_timestamp_local',
                       'max_value_in_window','min_value_in_window']].head(20).reset_index(drop=True)
else:
    top_pairs = pd.DataFrame(columns=['cmdb_id','trace_name','global_p95','global_p5','total_points_in_window',
                                      'anomalies_above_p95_count','earliest_above_p95_timestamp_local',
                                      'anomalies_below_p5_count','earliest_below_p5_timestamp_local',
                                      'max_value_in_window','min_value_in_window'])

# 4B) Aggregate by (cmdb_id, timestamp_local): count distinct trace_name anomalies at that timestamp
if not anomalies_trace_df.empty:
    # distinct anomalous trace names per cmdb_id and timestamp
    anomalous_by_ts = (anomalies_trace_df.groupby(['cmdb_id','_ts_local'], dropna=False)
                       .agg(
                           anomaly_count = ('trace_name', lambda x: x.nunique())
                       )
                       .reset_index())

    # total trace features at that timestamp (distinct trace_name) from window_trace_df
    total_features = (window_trace_df.groupby(['cmdb_id','_ts_local'], dropna=False)
                      .agg(total_trace_features_at_timestamp = ('trace_name', lambda x: x.nunique()))
                      .reset_index())

    # top 5 trace names anomalous per group by severity
    def top5_trace_names(g):
        # g is anomalies_trace_df group by cmdb_id and _ts_local
        by_name = (g.groupby('trace_name', dropna=False)
                   .agg(sev = ('severity','max'))
                   .reset_index()
                   .sort_values('sev', ascending=False))
        top5 = by_name['trace_name'].astype(str).head(5).tolist()
        return ",".join(top5)

    top5 = (anomalies_trace_df.groupby(['cmdb_id','_ts_local'], dropna=False)
            .apply(top5_trace_names)
            .reset_index()
            .rename(columns={0:'top_5_trace_names_anomalous'}))

    # earliest anomaly timestamp per cmdb_id
    earliest_per_cmdb = (anomalies_trace_df.groupby('cmdb_id', dropna=False)['_ts_local'].min().reset_index().rename(columns={'_ts_local':'earliest_trace_anomaly_timestamp_local'}))

    anomalies_by_ts = anomalous_by_ts.merge(total_features, on=['cmdb_id','_ts_local'], how='left')
    anomalies_by_ts = anomalies_by_ts.merge(top5, on=['cmdb_id','_ts_local'], how='left')

    # merge earliest timestamp per cmdb_id for convenience
    anomalies_by_ts = anomalies_by_ts.merge(earliest_per_cmdb, on='cmdb_id', how='left')

    # rename _ts_local to timestamp_local
    anomalies_by_ts = anomalies_by_ts.rename(columns={'_ts_local':'timestamp_local'})

    # prioritization: flag if top_5 contains from_frontend or cmdb is downstream
    anomalies_by_ts['has_from_frontend'] = anomalies_by_ts['top_5_trace_names_anomalous'].str.contains('from_frontend', na=False)
    anomalies_by_ts['is_downstream_cmdb'] = anomalies_by_ts['cmdb_id'].astype(str).apply(lambda x: any(x.startswith(s) for s in downstream_services))

    # sort by anomaly_count desc, prioritizing from_frontend and downstream
    anomalies_by_ts = anomalies_by_ts.sort_values(['anomaly_count','has_from_frontend','is_downstream_cmdb'], ascending=[False, False, False])

    top_ts = anomalies_by_ts[['cmdb_id','timestamp_local','anomaly_count','total_trace_features_at_timestamp','top_5_trace_names_anomalous','earliest_trace_anomaly_timestamp_local']].head(20).reset_index(drop=True)
else:
    top_ts = pd.DataFrame(columns=['cmdb_id','timestamp_local','anomaly_count','total_trace_features_at_timestamp','top_5_trace_names_anomalous','earliest_trace_anomaly_timestamp_local'])

# per-cmdb summary (top 10): total_trace_anomaly_count_in_window and earliest_trace_anomaly_timestamp_local
if not anomalies_trace_df.empty:
    per_cmdb_trace = (anomalies_trace_df.groupby('cmdb_id', dropna=False)
                      .agg(total_trace_anomaly_count_in_window = ('trace_name','count'),
                           earliest_trace_anomaly_timestamp_local = ('_ts_local','min'))
                      .reset_index()
                      .sort_values('total_trace_anomaly_count_in_window', ascending=False)
                      .head(10)
                      .reset_index(drop=True))
else:
    per_cmdb_trace = pd.DataFrame(columns=['cmdb_id','total_trace_anomaly_count_in_window','earliest_trace_anomaly_timestamp_local'])

# store intermediate tables for later correlation
_global_thresh_trace = global_thresh_trace
_window_trace_df = window_trace_df
_anomalies_trace_df = anomalies_trace_df
_pairs_trace_agg = pairs if 'pairs' in locals() else pd.DataFrame()
_anomalies_by_ts_trace = anomalies_by_ts if 'anomalies_by_ts' in locals() else pd.DataFrame()
_per_cmdb_trace_summary = per_cmdb_trace

# final compact outputs
top_pairs, top_ts, per_cmdb_trace

```
Out[4]:
```
# Summarize the trace analysis results in plain English (compact)
summary = (
    "Trace anomaly summary (global P95/P5 computed on full series; window 2022-03-21 03:00–03:30 Asia/Shanghai):\n\n"
    "1) Method: global P95/P5 were computed from the entire trace.csv. The window was filtered to 03:00–03:30 and rows were flagged using rules: "
    "duration/error_rate → anomaly if > global P95; row_count → anomaly if < global P5 (traffic drop).\n\n"
    "2) (cmdb_id, trace_name) hotspots (top examples):\n"
    "- Many top anomalies are 'trace.from_frontend-*' metrics for adservice instances (adservice-0/1/2). These show duration spikes (above P95) and paired row_count drops (below P5).\n"
    "- Example pattern: adservice-0 trace.from_frontend-0.duration_mean — 30 points in window, 2 above P95 (earliest 03:10), max ~0.000411; paired row_count had 2 drops (earliest 03:11).\n\n"
    "3) Timestamp-level hotspots (top examples):\n"
    "- frontend-0 @ 2022-03-21 03:12:00+08:00 — anomaly_count=11 (total trace features at that minute: 76).\n"
    "- frontend-2 @ 2022-03-21 03:18:00+08:00 — anomaly_count=11 (total features: 88).\n"
    "- Several checkoutservice timestamps appear as high-count hotspots (e.g., checkoutservice-1 @03:12 with 10 anomalies; checkoutservice-2 @03:07/03:13 with 6–7 anomalies).\n"
    "- Many hotspot timestamps list trace-to/downstream metrics (to productcatalogservice, to currencyservice, to adservice) among top-5 anomalous traces.\n\n"
    "4) Per-cmdb summary (top counts):\n"
    "- frontend-2: 112 trace anomalies in window (earliest 03:00:00+08:00)\n"
    "- frontend-1: 101 anomalies (earliest 03:00:00+08:00)\n"
    "- frontend-0: 99 anomalies (earliest 03:00:00+08:00)\n"
    "- checkoutservice-1: 53, checkoutservice-2: 45, frontend2-0: 45, checkoutservice2-0: 44, checkoutservice-0: 44, productcatalogservice-0: 44, productcatalogservice-1: 40.\n\n"
    "5) Interpretation and prioritization:\n"
    "- Anomalies are concentrated on traces originating from frontends and on traces involving downstream services (checkoutservice, productcatalogservice, adservice, etc.).\n"
    "- Common patterns: duration spikes at frontends/receivers and simultaneous row_count drops (traffic drops) for downstream calls — this suggests request latency and throughput issues starting at frontend traces and affecting downstream services.\n"
    "- Earliest anomalies for many components occur at or near 03:00:00+08:00, with multiple high-count minutes later in the window (notably ~03:11–03:18 and 03:26).\n\n"
    "Recommendation: focus investigation on frontend-origin traces at the hotspot timestamps (03:11–03:18 and specific minutes listed above) and on downstream services called by frontends (productcatalogservice, checkoutservice, adservice)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                           trace_name  global_p95  global_p5  total_points_in_window  anomalies_above_p95_count earliest_above_p95_timestamp_local  anomalies_below_p5_count earliest_below_p5_timestamp_local  max_value_in_window  min_value_in_window
0   adservice-0  trace.from_frontend-0.duration_mean    0.000025   0.000012                      30                          2          2022-03-21 03:10:00+08:00                         0                               NaT             0.000411             0.000012
1   adservice-0   trace.from_frontend-0.duration_p95    0.000063   0.000014                      30                          2          2022-03-21 03:10:00+08:00                         0                               NaT             0.001363             0.000014
2   adservice-0      trace.from_frontend-0.row_count    7.000000   3.450000                      30                          0                                NaT                         2         2022-03-21 03:11:00+08:00             8.000000             3.000000
3   adservice-0   trace.from_frontend-1.duration_p95    0.000021   0.000013                      30                          2          2022-03-21 03:24:00+08:00                         0                               NaT             0.000039             0.000013
4   adservice-0      trace.from_frontend-1.row_count    7.000000   3.450000                      30                          0                                NaT                         2         2022-03-21 03:01:00+08:00             7.000000             3.000000
5   adservice-0  trace.from_frontend-2.duration_mean    0.000017   0.000012                      30                          2          2022-03-21 03:11:00+08:00                         0                               NaT             0.000031             0.000012
6   adservice-0   trace.from_frontend-2.duration_p95    0.000027   0.000014                      30                          2          2022-03-21 03:11:00+08:00                         0                               NaT             0.000112             0.000013
7   adservice-0      trace.from_frontend-2.row_count   10.550000   5.450000                      30                          0                                NaT                         2         2022-03-21 03:08:00+08:00            12.000000             4.000000
8   adservice-1   trace.from_frontend-0.duration_p95    0.000023   0.000012                      30                          2          2022-03-21 03:07:00+08:00                         0                               NaT             0.000026             0.000012
9   adservice-1      trace.from_frontend-0.row_count    7.000000   3.450000                      30                          0                                NaT                         2         2022-03-21 03:11:00+08:00             8.000000             3.000000
10  adservice-1  trace.from_frontend-1.duration_mean    0.000018   0.000012                      30                          2          2022-03-21 03:13:00+08:00                         0                               NaT             0.000020             0.000012
11  adservice-1   trace.from_frontend-1.duration_p95    0.000029   0.000013                      30                          2          2022-03-21 03:13:00+08:00                         0                               NaT             0.000044             0.000012
12  adservice-1   trace.from_frontend-2.duration_p95    0.000030   0.000013                      30                          2          2022-03-21 03:00:00+08:00                         0                               NaT             0.000030             0.000013
13  adservice-1      trace.from_frontend-2.row_count   10.550000   5.450000                      30                          0                                NaT                         2         2022-03-21 03:06:00+08:00            12.000000             5.000000
14  adservice-2  trace.from_frontend-0.duration_mean    0.000015   0.000012                      30                          2          2022-03-21 03:02:00+08:00                         0                               NaT             0.000016             0.000011
15  adservice-2   trace.from_frontend-0.duration_p95    0.000019   0.000013                      30                          2          2022-03-21 03:00:00+08:00                         0                               NaT             0.000024             0.000012
16  adservice-2  trace.from_frontend-1.duration_mean    0.000016   0.000012                      30                          2          2022-03-21 03:15:00+08:00                         0                               NaT             0.000016             0.000012
17  adservice-2   trace.from_frontend-1.duration_p95    0.000021   0.000013                      30                          2          2022-03-21 03:15:00+08:00                         0                               NaT             0.000024             0.000013
18  adservice-2  trace.from_frontend-2.duration_mean    0.000015   0.000012                      30                          2          2022-03-21 03:20:00+08:00                         0                               NaT             0.000016             0.000012
19  adservice-2   trace.from_frontend-2.duration_p95    0.000020   0.000013                      30                          2          2022-03-21 03:20:00+08:00                         0                               NaT             0.000029             0.000013,                     cmdb_id           timestamp_local  anomaly_count  total_trace_features_at_timestamp                        top_5_trace_names_anomalous earliest_trace_anomaly_timestamp_local
0                frontend-0 2022-03-21 03:12:00+08:00             11                                 76  trace.to_productcatalogservice-2.row_count,tra...              2022-03-21 03:00:00+08:00
1                frontend-2 2022-03-21 03:18:00+08:00             11                                 88  trace.self.row_count,trace.to_productcatalogse...              2022-03-21 03:00:00+08:00
2         checkoutservice-1 2022-03-21 03:12:00+08:00             10                                 80  trace.to_productcatalogservice-2.duration_p95,...              2022-03-21 03:00:00+08:00
3                frontend-1 2022-03-21 03:17:00+08:00              9                                 84  trace.to_currencyservice-2.row_count,trace.to_...              2022-03-21 03:00:00+08:00
4                frontend-1 2022-03-21 03:22:00+08:00              9                                 80  trace.self.row_count,trace.to_productcatalogse...              2022-03-21 03:00:00+08:00
5                frontend-2 2022-03-21 03:07:00+08:00              9                                 72  trace.self.row_count,trace.from_root.row_count...              2022-03-21 03:00:00+08:00
6         checkoutservice-1 2022-03-21 03:04:00+08:00              8                                 56  trace.to_productcatalogservice-1.duration_mean...              2022-03-21 03:00:00+08:00
7        checkoutservice2-0 2022-03-21 03:19:00+08:00              8                                 36  trace.from_root.row_count,trace.self.row_count...              2022-03-21 03:00:00+08:00
8                frontend-0 2022-03-21 03:11:00+08:00              8                                 80  trace.to_adservice-2.row_count,trace.to_curren...              2022-03-21 03:00:00+08:00
9                frontend-0 2022-03-21 03:27:00+08:00              8                                 84  trace.to_productcatalogservice-0.duration_p95,...              2022-03-21 03:00:00+08:00
10               frontend-2 2022-03-21 03:16:00+08:00              8                                 88  trace.to_productcatalogservice-0.row_count,tra...              2022-03-21 03:00:00+08:00
11  productcatalogservice-1 2022-03-21 03:01:00+08:00              7                                 36  trace.from_recommendationservice-1.row_count,t...              2022-03-21 03:01:00+08:00
12        checkoutservice-0 2022-03-21 03:18:00+08:00              7                                 56  trace.to_currencyservice-2.duration_mean,trace...              2022-03-21 03:03:00+08:00
13               frontend-0 2022-03-21 03:17:00+08:00              7                                 88  trace.to_cartservice-2.duration_p95,trace.to_c...              2022-03-21 03:00:00+08:00
14               frontend-0 2022-03-21 03:25:00+08:00              7                                 84  trace.from_root.duration_p95,trace.from_root.d...              2022-03-21 03:00:00+08:00
15               frontend-1 2022-03-21 03:01:00+08:00              7                                 84  trace.to_adservice-0.row_count,trace.to_recomm...              2022-03-21 03:00:00+08:00
16              frontend2-0 2022-03-21 03:15:00+08:00              7                                 36  trace.self.row_count,trace.to_currencyservice2...              2022-03-21 03:00:00+08:00
17        checkoutservice-2 2022-03-21 03:07:00+08:00              6                                 92  trace.from_frontend-0.duration_mean,trace.from...              2022-03-21 03:01:00+08:00
18        checkoutservice-2 2022-03-21 03:13:00+08:00              6                                 80  trace.from_frontend-1.duration_mean,trace.from...              2022-03-21 03:01:00+08:00
19  recommendationservice-0 2022-03-21 03:12:00+08:00              6                                 28  trace.from_frontend-0.row_count,trace.from_fro...              2022-03-21 03:00:00+08:00,                    cmdb_id  total_trace_anomaly_count_in_window earliest_trace_anomaly_timestamp_local
0               frontend-2                                  112              2022-03-21 03:00:00+08:00
1               frontend-1                                  101              2022-03-21 03:00:00+08:00
2               frontend-0                                   99              2022-03-21 03:00:00+08:00
3        checkoutservice-1                                   53              2022-03-21 03:00:00+08:00
4        checkoutservice-2                                   45              2022-03-21 03:01:00+08:00
5              frontend2-0                                   45              2022-03-21 03:00:00+08:00
6       checkoutservice2-0                                   44              2022-03-21 03:00:00+08:00
7        checkoutservice-0                                   44              2022-03-21 03:03:00+08:00
8  productcatalogservice-0                                   44              2022-03-21 03:00:00+08:00
9  productcatalogservice-1                                   40              2022-03-21 03:01:00+08:00)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# reuse existing dataframes if present, otherwise load
if 'error_df' not in globals():
    error_df = pd.read_csv('error_logs.csv')
if 'log_df' not in globals():
    log_df = pd.read_csv('log.csv')

# ensure timestamp numeric and localized for error_df
error_df['timestamp'] = pd.to_numeric(error_df['timestamp'], errors='coerce').astype('Int64')
if '_ts_local' not in error_df.columns:
    error_dt = pd.to_datetime(error_df['timestamp'].astype('int64'), unit='s', utc=True)
    error_df['_ts_local'] = error_dt.dt.tz_convert(tz)

# 1) error_logs.csv aggregation by cmdb_id (top 10)
error_agg = (
    error_df
    .groupby('cmdb_id', dropna=False)['_ts_local']
    .agg(count='count', earliest_timestamp_local='min', latest_timestamp_local='max')
    .reset_index()
    .sort_values('count', ascending=False)
)
error_agg_top10 = error_agg.head(10).reset_index(drop=True)

# sample up to 5 raw error_log rows for the top 3 cmdb_id (timestamp, cmdb_id, message)
top3_cmdbs = error_agg_top10['cmdb_id'].head(3).tolist()
error_samples_top3 = (
    error_df[error_df['cmdb_id'].isin(top3_cmdbs)]
    .loc[:, ['_ts_local','cmdb_id','message']]
    .sort_values(['cmdb_id','_ts_local'])
    .groupby('cmdb_id', group_keys=False)
    .head(5)
    .rename(columns={'_ts_local':'timestamp_local'})
    .reset_index(drop=True)
)

# ensure timestamp column name for display consistency
error_agg_top10 = error_agg_top10[['cmdb_id','count','earliest_timestamp_local','latest_timestamp_local']]

# 2) log.csv frontend error analysis within incident window
# ensure timestamp/local column exists
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
if '_ts_local' not in log_df.columns:
    log_dt = pd.to_datetime(log_df['timestamp'].astype('int64'), unit='s', utc=True)
    log_df['_ts_local'] = log_dt.dt.tz_convert(tz)

# filter window inclusive
start = pd.Timestamp('2022-03-21 03:00:00', tz=tz)
end   = pd.Timestamp('2022-03-21 03:30:00', tz=tz)
log_window = log_df[(log_df['_ts_local'] >= start) & (log_df['_ts_local'] <= end)].copy()

# filter frontends (cmdb_id starting with 'frontend')
is_frontend = log_window['cmdb_id'].astype(str).str.startswith('frontend')
log_window_frontend = log_window[is_frontend].copy()

# focus on log_name == 'log.error_count'
frontend_error_logs = log_window_frontend[log_window_frontend['log_name'] == 'log.error_count'].copy()
frontend_error_logs['value'] = pd.to_numeric(frontend_error_logs['value'], errors='coerce').fillna(0.0)

# per-cmdb aggregation: total_error_count_sum, number_of_minutes_with_error, earliest_minute_with_error (value>0)
def first_nonzero_ts(s):
    nonzero = s[s['value'] > 0]
    if nonzero.empty:
        return pd.NaT
    return nonzero['_ts_local'].min()

frontend_error_agg = (
    frontend_error_logs
    .groupby('cmdb_id', dropna=False)
    .agg(
        total_error_count_sum = ('value','sum'),
        number_of_minutes_with_error = ('value', lambda x: int((x>0).sum())),
    )
    .reset_index()
)

# compute earliest_minute_with_error separately (need timestamp)
earliest_errors = (
    frontend_error_logs[frontend_error_logs['value'] > 0]
    .groupby('cmdb_id', dropna=False)['_ts_local']
    .min()
    .reset_index()
    .rename(columns={'_ts_local':'earliest_minute_with_error'})
)
frontend_error_agg = frontend_error_agg.merge(earliest_errors, on='cmdb_id', how='left')

# top 10 frontends by number_of_minutes_with_error desc
frontend_error_agg_top10 = frontend_error_agg.sort_values(['number_of_minutes_with_error','total_error_count_sum'], ascending=[False,False]).head(10).reset_index(drop=True)

# rows where value>0 across these frontends (limit 20 rows) with timestamp_local, cmdb_id, value
top_frontends_list = frontend_error_agg_top10['cmdb_id'].tolist()
frontend_positive_rows = (
    frontend_error_logs[
        (frontend_error_logs['cmdb_id'].isin(top_frontends_list)) & (frontend_error_logs['value'] > 0)
    ]
    .loc[:, ['_ts_local','cmdb_id','value']]
    .rename(columns={'_ts_local':'timestamp_local'})
    .sort_values(['cmdb_id','timestamp_local'])
    .head(20)
    .reset_index(drop=True)
)

# store intermediate tables for later correlation
_error_agg_full = error_agg
_error_samples_top3 = error_samples_top3
_log_window_frontend = log_window_frontend
_frontend_error_logs = frontend_error_logs
_frontend_error_agg_full = frontend_error_agg

# Final compact outputs (bounded)
error_agg_top10, error_samples_top3, frontend_error_agg_top10, frontend_positive_rows

```
Out[5]:
```
```python
# Compact plain-English summary of error_logs.csv and log.csv analysis (results from prior execution)
summary = (
    "Summary of error and log analysis (window 2022-03-21 03:00–03:30 Asia/Shanghai):\n\n"
    "1) error_logs.csv (top CMDBs):\n"
    "- frontend-2: 232 error log entries (earliest 2022-03-21 03:00:05, latest 03:29:53)\n"
    "- frontend-0: 158 entries (earliest 03:00:07, latest 03:29:31)\n"
    "- frontend-1: 154 entries (earliest 03:00:06, latest 03:29:56)\n"
    "  Sample messages for these frontends are repeated warnings of the form: \"severity: warning, message: failed to retrieve...\".\n\n"
    "2) log.csv (frontend log.error_count within window):\n"
    "- All three frontends show persistent error activity across the entire 30-minute window: frontend-2 sum=232 (errors present all 30 minutes), frontend-0 sum=158 (30 minutes), frontend-1 sum=154 (30 minutes).\n"
    "- Per-minute samples (frontends) show nonzero log.error_count values every minute (examples listed for frontend-0 with minute-by-minute counts such as 4,6,5, ... up to 8).\n\n"
    "Interpretation / next steps:\n"
    "- Frontend instances are the primary sources of logged warnings and elevated error counts during the incident window.\n"
    "- The error message wording ('failed to retrieve...') suggests repeated retrieval failures; correlate these front-end logs with the previously computed trace and metric anomalies (frontend-origin traces and downstream latency/row_count drops) for root-cause investigation.\n"
    "- Prioritize investigating frontend-2, frontend-0, and frontend-1 and their downstream calls (e.g., productcatalogservice, checkoutservice, adservice) at hotspot timestamps found in trace/metric analysis.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  count  earliest_timestamp_local    latest_timestamp_local
0  frontend-2    232 2022-03-21 03:00:05+08:00 2022-03-21 03:29:53+08:00
1  frontend-0    158 2022-03-21 03:00:07+08:00 2022-03-21 03:29:31+08:00
2  frontend-1    154 2022-03-21 03:00:06+08:00 2022-03-21 03:29:56+08:00,              timestamp_local     cmdb_id                                            message
0  2022-03-21 03:00:07+08:00  frontend-0  severity: warning, message: failed to retrieve...
1  2022-03-21 03:00:17+08:00  frontend-0  severity: warning, message: failed to retrieve...
2  2022-03-21 03:00:29+08:00  frontend-0  severity: warning, message: failed to retrieve...
3  2022-03-21 03:00:43+08:00  frontend-0  severity: warning, message: failed to retrieve...
4  2022-03-21 03:01:04+08:00  frontend-0  severity: warning, message: failed to retrieve...
5  2022-03-21 03:00:06+08:00  frontend-1  severity: warning, message: failed to retrieve...
6  2022-03-21 03:00:14+08:00  frontend-1  severity: warning, message: failed to retrieve...
7  2022-03-21 03:00:24+08:00  frontend-1  severity: warning, message: failed to retrieve...
8  2022-03-21 03:00:34+08:00  frontend-1  severity: warning, message: failed to retrieve...
9  2022-03-21 03:00:48+08:00  frontend-1  severity: warning, message: failed to retrieve...
10 2022-03-21 03:00:05+08:00  frontend-2  severity: warning, message: failed to retrieve...
11 2022-03-21 03:00:10+08:00  frontend-2  severity: warning, message: failed to retrieve...
12 2022-03-21 03:00:14+08:00  frontend-2  severity: warning, message: failed to retrieve...
13 2022-03-21 03:00:21+08:00  frontend-2  severity: warning, message: failed to retrieve...
14 2022-03-21 03:00:25+08:00  frontend-2  severity: warning, message: failed to retrieve...,       cmdb_id  total_error_count_sum  number_of_minutes_with_error earliest_minute_with_error
0  frontend-2                  232.0                            30  2022-03-21 03:00:00+08:00
1  frontend-0                  158.0                            30  2022-03-21 03:00:00+08:00
2  frontend-1                  154.0                            30  2022-03-21 03:00:00+08:00,              timestamp_local     cmdb_id  value
0  2022-03-21 03:00:00+08:00  frontend-0    4.0
1  2022-03-21 03:01:00+08:00  frontend-0    6.0
2  2022-03-21 03:02:00+08:00  frontend-0    5.0
3  2022-03-21 03:03:00+08:00  frontend-0    5.0
4  2022-03-21 03:04:00+08:00  frontend-0    6.0
5  2022-03-21 03:05:00+08:00  frontend-0    7.0
6  2022-03-21 03:06:00+08:00  frontend-0    5.0
7  2022-03-21 03:07:00+08:00  frontend-0    6.0
8  2022-03-21 03:08:00+08:00  frontend-0    5.0
9  2022-03-21 03:09:00+08:00  frontend-0    4.0
10 2022-03-21 03:10:00+08:00  frontend-0    7.0
11 2022-03-21 03:11:00+08:00  frontend-0    3.0
12 2022-03-21 03:12:00+08:00  frontend-0    6.0
13 2022-03-21 03:13:00+08:00  frontend-0    4.0
14 2022-03-21 03:14:00+08:00  frontend-0    7.0
15 2022-03-21 03:15:00+08:00  frontend-0    4.0
16 2022-03-21 03:16:00+08:00  frontend-0    8.0
17 2022-03-21 03:17:00+08:00  frontend-0    5.0
18 2022-03-21 03:18:00+08:00  frontend-0    6.0
19 2022-03-21 03:19:00+08:00  frontend-0    6.0)```
```

In [ ]:
# In[6]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[6]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes if present
tz = pytz.timezone('Asia/Shanghai')

# load if not present (per task files exist)
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')
if 'log_df' not in globals():
    log_df = pd.read_csv('log.csv')
if 'error_df' not in globals():
    error_df = pd.read_csv('error_logs.csv')

# Ensure timestamp local columns (UTC+8) exist
def ensure_local(df):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    if '_ts_local' not in df.columns:
        dt = pd.to_datetime(df['timestamp'].astype('int64'), unit='s', utc=True)
        df['_ts_local'] = dt.dt.tz_convert(tz)
    return df

metric_df = ensure_local(metric_df)
trace_df = ensure_local(trace_df)
log_df = ensure_local(log_df)
error_df = ensure_local(error_df)

# Incident window bounds
start = pd.Timestamp('2022-03-21 03:00:00', tz=tz)
end   = pd.Timestamp('2022-03-21 03:30:00', tz=tz)

# Frontend list
frontends = ['frontend-0','frontend-1','frontend-2','frontend2-0']

# 1) Metric global thresholds (full series) per (cmdb_id,kpi_name)
if '_metric_global_thresh' in globals():
    _metric_global_thresh = globals()['_metric_global_thresh']
else:
    _metric_global_thresh = (
        metric_df
        .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
        .agg(global_p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan,
             global_p5 =lambda x: float(x.dropna().quantile(0.05)) if x.dropna().size>0 else np.nan)
        .reset_index()
    )

# Filter metric to window and frontend cmdbs
metric_window = metric_df[(metric_df['_ts_local'] >= start) & (metric_df['_ts_local'] <= end) & (metric_df['cmdb_id'].isin(frontends))].copy()
# merge thresholds
metric_window = metric_window.merge(_metric_global_thresh, on=['cmdb_id','kpi_name'], how='left')
# mark anomalies: value > global_p95 or value < global_p5
metric_window['anomalous_metric'] = (metric_window['value'] > metric_window['global_p95']) | (metric_window['value'] < metric_window['global_p5'])
# floor timestamp to minute (safe)
metric_window['ts_min'] = metric_window['_ts_local'].dt.floor('T')

# Aggregate metric anomalies per (ts_min, cmdb_id): count distinct kpi_name anomalous at that minute
metric_anoms_by_min = (
    metric_window[metric_window['anomalous_metric']]
    .groupby(['cmdb_id','ts_min'], dropna=False)
    .agg(metric_anomaly_count = ('kpi_name', lambda x: x.nunique()))
    .reset_index()
    .rename(columns={'ts_min':'timestamp_local'})
)
# ensure int
if not metric_anoms_by_min.empty:
    metric_anoms_by_min['metric_anomaly_count'] = metric_anoms_by_min['metric_anomaly_count'].astype(int)

# 2) Trace global thresholds (full series) per (cmdb_id,trace_name)
if '_trace_global_thresh' in globals():
    _trace_global_thresh = globals()['_trace_global_thresh']
else:
    _trace_global_thresh = (
        trace_df
        .groupby(['cmdb_id','trace_name'], dropna=False)['value']
        .agg(global_p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan,
             global_p5 =lambda x: float(x.dropna().quantile(0.05)) if x.dropna().size>0 else np.nan)
        .reset_index()
    )

# Filter trace to window and frontends
trace_window = trace_df[(trace_df['_ts_local'] >= start) & (trace_df['_ts_local'] <= end) & (trace_df['cmdb_id'].isin(frontends))].copy()
trace_window = trace_window.merge(_trace_global_thresh, on=['cmdb_id','trace_name'], how='left')
trace_window['ts_min'] = trace_window['_ts_local'].dt.floor('T')

# Mark trace anomalies per rules:
# duration* => anomalous if value > global_p95
# error_rate => anomalous if value > global_p95
# row_count => anomalous if value < global_p5
trace_window['is_duration'] = trace_window['trace_name'].str.contains('duration', na=False)
trace_window['is_error_rate'] = trace_window['trace_name'].str.contains('error_rate', na=False)
trace_window['is_row_count'] = trace_window['trace_name'].str.contains('row_count', na=False)

trace_window['anomalous_trace'] = False
mask = trace_window['value'].notna()
trace_window.loc[mask & trace_window['is_duration'], 'anomalous_trace'] = trace_window.loc[mask & trace_window['is_duration'], 'value'] > trace_window.loc[mask & trace_window['is_duration'], 'global_p95']
trace_window.loc[mask & trace_window['is_error_rate'], 'anomalous_trace'] = trace_window.loc[mask & trace_window['is_error_rate'], 'value'] > trace_window.loc[mask & trace_window['is_error_rate'], 'global_p95']
trace_window.loc[mask & trace_window['is_row_count'], 'anomalous_trace'] = trace_window.loc[mask & trace_window['is_row_count'], 'value'] < trace_window.loc[mask & trace_window['is_row_count'], 'global_p5']

# Aggregate trace anomalies per minute per frontend: distinct trace_name count
trace_anoms_by_min = (
    trace_window[trace_window['anomalous_trace']]
    .groupby(['cmdb_id','ts_min'], dropna=False)
    .agg(trace_anomaly_count = ('trace_name', lambda x: x.nunique()))
    .reset_index()
    .rename(columns={'ts_min':'timestamp_local'})
)
if not trace_anoms_by_min.empty:
    trace_anoms_by_min['trace_anomaly_count'] = trace_anoms_by_min['trace_anomaly_count'].astype(int)

# 3) log.csv: filter to window and frontends; compute per-minute sum for log.error_count
log_window = log_df[(log_df['_ts_local'] >= start) & (log_df['_ts_local'] <= end) & (log_df['cmdb_id'].isin(frontends))].copy()
log_window['ts_min'] = log_window['_ts_local'].dt.floor('T')
log_window['value'] = pd.to_numeric(log_window['value'], errors='coerce').fillna(0.0)
log_error_counts = (
    log_window[log_window['log_name']=='log.error_count']
    .groupby(['cmdb_id','ts_min'], dropna=False)
    .agg(log_error_count = ('value','sum'))
    .reset_index()
    .rename(columns={'ts_min':'timestamp_local'})
)
if not log_error_counts.empty:
    log_error_counts['log_error_count'] = log_error_counts['log_error_count'].astype(int)

# 4) error_logs.csv: filter to window and frontends; per-minute counts
error_window = error_df[(error_df['_ts_local'] >= start) & (error_df['_ts_local'] <= end) & (error_df['cmdb_id'].isin(frontends))].copy()
# floor to minute
error_window['ts_min'] = error_window['_ts_local'].dt.floor('T')
error_log_counts = (
    error_window.groupby(['cmdb_id','ts_min'], dropna=False)
    .size()
    .reset_index(name='error_log_count')
    .rename(columns={'ts_min':'timestamp_local'})
)
if not error_log_counts.empty:
    error_log_counts['error_log_count'] = error_log_counts['error_log_count'].astype(int)

# 5) Merge aggregates per (timestamp_local, cmdb_id)
# Create full index of minutes for frontends in window (to ensure missing minutes are zero)
all_minutes = pd.date_range(start=start, end=end, freq='T', tz=tz)
idx = pd.MultiIndex.from_product([frontends, all_minutes], names=['cmdb_id','timestamp_local'])
agg_base = pd.DataFrame(index=idx).reset_index()

# left-merge each aggregated table
agg = agg_base.merge(metric_anoms_by_min, on=['cmdb_id','timestamp_local'], how='left')
agg = agg.merge(trace_anoms_by_min, on=['cmdb_id','timestamp_local'], how='left')
agg = agg.merge(log_error_counts, on=['cmdb_id','timestamp_local'], how='left')
agg = agg.merge(error_log_counts, on=['cmdb_id','timestamp_local'], how='left')

# fill NaNs with zeros
agg['metric_anomaly_count'] = agg['metric_anomaly_count'].fillna(0).astype(int)
agg['trace_anomaly_count'] = agg['trace_anomaly_count'].fillna(0).astype(int)
agg['log_error_count'] = agg['log_error_count'].fillna(0).astype(int)
agg['error_log_count'] = agg['error_log_count'].fillna(0).astype(int)

# Keep only rows where any of the four counts > 0
agg['total_events'] = agg['metric_anomaly_count'] + agg['trace_anomaly_count'] + agg['log_error_count'] + agg['error_log_count']
correlation_rows = agg[agg['total_events'] > 0].copy()

# Sort by total_events desc, then timestamp asc, cmdb_id
correlation_rows = correlation_rows.sort_values(['total_events','timestamp_local'], ascending=[False,True]).reset_index(drop=True)

# Top table: limit to top 20 rows, select required columns
correlation_top20 = correlation_rows[['timestamp_local','cmdb_id','metric_anomaly_count','trace_anomaly_count','log_error_count','error_log_count']].head(20).copy()

# 6) Earliest-full-correlation per frontend: earliest timestamp_local where metric_anomaly_count>0 AND trace_anomaly_count>0 AND error_log_count>0
earliest_full = []
for fe in frontends:
    df_fe = correlation_rows[(correlation_rows['cmdb_id']==fe) & (correlation_rows['metric_anomaly_count']>0) & (correlation_rows['trace_anomaly_count']>0) & (correlation_rows['error_log_count']>0)]
    if not df_fe.empty:
        ts = df_fe['timestamp_local'].min()
    else:
        ts = pd.NaT
    earliest_full.append({'cmdb_id':fe, 'earliest_full_correlation_timestamp_local': ts})

earliest_full_df = pd.DataFrame(earliest_full)

# Convert NaT to None for clarity (but keep tz-aware types where present)
earliest_full_df['earliest_full_correlation_timestamp_local'] = earliest_full_df['earliest_full_correlation_timestamp_local'].where(earliest_full_df['earliest_full_correlation_timestamp_local'].notna(), pd.NaT)

# Store intermediate tables for later steps
_metric_window_for_next = metric_window
_trace_window_for_next = trace_window
_log_window_for_next = log_window
_error_window_for_next = error_window
_metric_anoms_by_min_for_next = metric_anoms_by_min
_trace_anoms_by_min_for_next = trace_anoms_by_min
_log_error_counts_for_next = log_error_counts
_error_log_counts_for_next = error_log_counts
_correlation_full_table_for_next = correlation_rows

# Final outputs (compact, bounded)
correlation_top20, earliest_full_df

```
Out[7]:
```
summary = (
    "Minute-level correlation for frontends (03:00–03:30 Asia/Shanghai):\n\n"
    "What was built: per-minute aggregates for frontend-0, frontend-1, frontend-2, frontend2-0 counting\n"
    "- metric_anomaly_count (metrics outside global P5/P95),\n"
    "- trace_anomaly_count (trace rules: duration/error_rate high, row_count low),\n"
    "- log_error_count (sum of log.error_count), and\n"
    "- error_log_count (count of error_log rows).\n\n"
    "Top correlated minutes (examples from top-20 sorted by total events):\n"
    "1) frontend-1 @ 2022-03-21 03:16 — metric=38, trace=4, log=6, error_log=6 (overall highest total)\n"
    "2) frontend-0 @ 2022-03-21 03:17 — metric=33, trace=7, log=5, error_log=5\n"
    "3) frontend-2 @ 2022-03-21 03:00 — metric=22, trace=5, log=11, error_log=11 (heavy log/error activity at window start)\n"
    "4) frontend-0 @ 2022-03-21 03:12 — metric=26, trace=11, log=6, error_log=6\n"
    "5) frontend-2 @ 2022-03-21 03:07 and 03:11 — metric and trace spikes with matching log/error counts\n"
    "Also notable: frontend2-0 @ 03:27 shows metric_anomaly_count=33 but log_error_count and error_log_count = 0 (metric-only anomalies).\n\n"
    "Earliest full correlations (metric>0 AND trace>0 AND error_log>0):\n"
    "- frontend-0: 2022-03-21 03:00:00+08:00\n"
    "- frontend-1: 2022-03-21 03:00:00+08:00\n"
    "- frontend-2: 2022-03-21 03:00:00+08:00\n"
    "- frontend2-0: none (no minute where all three conditions met)\n\n"
    "Interpretation / next steps:\n"
    "- Strong, simultaneous anomalies across metrics, traces, and logs exist for frontend-0/1/2 starting at the window start (03:00) and recurring at hotspot minutes around 03:11–03:18, 03:16–03:17, and ~03:26.\n"
    "- frontend2-0 shows metric-only anomalies at 03:27, suggesting either isolated metric issues or missing log/error captures.\n"
    "- Prioritize investigating frontends at the earliest correlated minute (03:00) and the high-impact hotspot minutes above; correlate these with downstream calls (productcatalogservice, checkoutservice, adservice) and service metrics/trace details."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             timestamp_local      cmdb_id  metric_anomaly_count  trace_anomaly_count  log_error_count  error_log_count
0  2022-03-21 03:16:00+08:00   frontend-1                    38                    4                6                6
1  2022-03-21 03:17:00+08:00   frontend-0                    33                    7                5                5
2  2022-03-21 03:00:00+08:00   frontend-2                    22                    5               11               11
3  2022-03-21 03:12:00+08:00   frontend-0                    26                   11                6                6
4  2022-03-21 03:07:00+08:00   frontend-2                    21                    9                9                9
5  2022-03-21 03:20:00+08:00   frontend-0                    34                    1                6                6
6  2022-03-21 03:16:00+08:00   frontend-2                    21                    8                8                8
7  2022-03-21 03:06:00+08:00   frontend-1                    29                    0                7                7
8  2022-03-21 03:18:00+08:00   frontend-2                    16                   11                8                8
9  2022-03-21 03:22:00+08:00   frontend-1                    24                    9                5                5
10 2022-03-21 03:01:00+08:00   frontend-2                    25                    2                7                7
11 2022-03-21 03:26:00+08:00   frontend-0                    24                    4                5                5
12 2022-03-21 03:02:00+08:00   frontend-1                    19                    6                6                6
13 2022-03-21 03:11:00+08:00   frontend-2                    10                    6               10               10
14 2022-03-21 03:09:00+08:00   frontend-2                     7                    4               12               12
15 2022-03-21 03:00:00+08:00   frontend-1                    16                    6                6                6
16 2022-03-21 03:12:00+08:00   frontend-2                    11                    4                9                9
17 2022-03-21 03:14:00+08:00   frontend-2                    14                    3                8                8
18 2022-03-21 03:27:00+08:00  frontend2-0                    33                    0                0                0
19 2022-03-21 03:29:00+08:00   frontend-0                    20                    6                3                3,        cmdb_id earliest_full_correlation_timestamp_local
0   frontend-0                 2022-03-21 03:00:00+08:00
1   frontend-1                 2022-03-21 03:00:00+08:00
2   frontend-2                 2022-03-21 03:00:00+08:00
3  frontend2-0                                       NaT)```
```